# 🎮 Mad Mario - DQN 训练交互式 Notebook

本 Notebook 提供了完整的 DQN 训练流程，替代命令行方式。所有配置均在下方单元格中完成，训练过程实时显示进度与图表。

**主要功能：**
- 单环境训练 / 多环境并行训练
- 单关卡 / 多关卡训练
- 实时训练指标可视化
- 模型评估与保存
- 断点续训
- 观看训练好的智能体玩游戏

In [ ]:
import os
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython import display


In [ ]:

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# ── 确认项目包可导入 ──
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from mad_mario.agent.checkpoint import CheckpointManager
from mad_mario.agent.mario import Mario
from mad_mario.config import (
    AgentConfig,
    AppConfig,
    ArtifactConfig,
    EnvConfig,
    TrainingConfig,
    build_env_name,
    compatible_save_root,
)
from mad_mario.env.factory import make_mario_env, make_vector_env
from mad_mario.logging.metrics import MetricLogger
from mad_mario.training.artifacts import create_artifacts
from mad_mario.training.evaluation import evaluate_agent
from mad_mario.training.loops import (
    close_vector_envs,
    get_replay_next_states,
    get_info_array,
    should_record,
    should_eval,
    should_save_step,
    rolling_mean,
)

# ── 检测硬件 ──
USE_CUDA = torch.cuda.is_available()
print(f"🔥  PyTorch 版本: {torch.__version__}")
print(f"💻  CUDA 可用: {USE_CUDA}")
if USE_CUDA:
    print(f"     GPU: {torch.cuda.get_device_name(0)}")
    print(f"     显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

---
## ⚙️ 训练配置

修改下方参数即可调整训练设置。每个参数都有中文注释说明。

In [ ]:
# ═══════════════════════════════════════════
#  环境配置
# ═══════════════════════════════════════════

ENV_CONFIG = dict(
    game="SuperMarioBros",       # 游戏名称
    world=1,                      # 世界/大关 (单关卡时起效)
    level=1,                      # 关卡 (单关卡时起效)
    version=0,                    # 环境版本
    movement="right_only",       # 动作空间: right_only | simple | complex

    # 多关卡训练 —— 设置后覆盖上方 world/level
    levels=None,                  # 如 ["1-1", "1-2", "1-3"]，或 None
    worlds=None,                  # 如 [1, 2]，与 level(s) 做笛卡尔积，或 None

    clip_rewards=True,            # 是否裁剪奖励
    reward_clip_value=1.0,        # 奖励裁剪范围 [-x, x]
    stuck_penalty_enabled=True,   # 卡住惩罚（卡住120步终止并扣分）
    stuck_max_steps=120,          # 判定卡住的连续无位移步数
    stuck_penalty=5.0,            # 卡住时额外扣除的奖励
    progress_reward_enabled=True, # 向右移动奖励
    progress_reward_scale=0.01,   # 每像素奖励系数
    normalize_observation=False,  # 归一化观测（保持 uint8 可省显存）
)

# ═══════════════════════════════════════════
#  智能体（网络 / 算法）配置
# ═══════════════════════════════════════════

AGENT_CONFIG = dict(
    replay_capacity=100_000,         # 经验回放池大小
    gamma=0.99,                      # 折扣因子 γ
    learning_rate=0.0000625,         # 初始学习率
    lr_warmup_steps=10_000,          # 学习率 warmup 步数
    lr_min_ratio=0.1,                # 最小学习率 = lr_min_ratio × learning_rate
    exploration_rate=1.0,            # epsilon 初始值
    exploration_decay_steps=2_000_000, # epsilon 线性衰减步数
    exploration_rate_min=0.02,       # epsilon 最小值
    gradient_clip_norm=10.0,         # 梯度裁剪范数
    n_step=3,                        # N-step return 步数
    noisy_std_init=0.5,              # NoisyNet 噪声标准差（设为 0 则用 epsilon-greedy）
    per_alpha=0.6,                   # PER 优先级指数 α
    per_beta_start=0.4,              # PER 重要性采样 β 初始值
    per_beta_frames=2_000_000,       # β 退火步数
    per_epsilon=1e-6,               # PER 最小优先级
)

# ═══════════════════════════════════════════
#  训练流程配置
# ═══════════════════════════════════════════

TRAINING_CONFIG = dict(
    episodes=500,                  # 训练回合数（可随时中断，支持续训）
    batch_size=32,                 # 每次学习的 batch 大小
    burnin=50_000,                 # 开始学习前收集的经验数量
    learn_every=4,                 # 每隔 N 步学习一次
    sync_every=8_000,             # 每隔 N 步同步 target 网络
    record_every=50,               # 每隔 N 回合打印/保存指标
    eval_every=100,               # 每隔 N 回合做一次贪心评估
    eval_episodes=3,               # 每次评估运行的回合数
)

# ═══════════════════════════════════════════
#  并行 / 存储配置
# ═══════════════════════════════════════════

VECTOR_ENV = False                  # 是否启用并行环境训练
NUM_ENVS = max(1, (os.cpu_count() or 2) // 2)  # 并行环境数（自动检测 CPU 核数）
SAVE_ROOT = Path("../checkpoints")    # checkpoint 根目录
FLAT_SAVE_ROOT = False              # 直接用根目录，不分子目录
KEEP_RUNS = False                   # 额外保留本次训练的 run 目录
RUN_NAME = None                     # 自定义 run 名称（None 则自动生成时间戳）
SAVE_EVERY = 500_000               # 每隔 N 步保存一次 checkpoint
RESUME = True                       # 是否从 latest checkpoint 恢复

print("✅ 配置加载完成")
print(f"    训练方式: {'🔄 并行' if VECTOR_ENV else '➡️ 单环境'}  回合数: {TRAINING_CONFIG['episodes']}")
print(f"    动作空间: {ENV_CONFIG['movement']}  " + (f"  关卡: {ENV_CONFIG['levels']}" if ENV_CONFIG['levels'] else f"  关卡: {ENV_CONFIG['world']}-{ENV_CONFIG['level']}"))
print(f"    保存路径: {SAVE_ROOT}")

---
## 🔧 构建配置对象

运行此单元格将上面的字典转换成框架内部使用的配置对象。

In [ ]:
# ── 展开关卡级别配置 ──
levels = ENV_CONFIG.get("levels")
worlds = ENV_CONFIG.get("worlds")

if worlds:
    expanded = []
    for w in worlds:
        raw = ENV_CONFIG.get("levels") or [str(ENV_CONFIG["level"])]
        for rl in raw:
            if "-" in rl:
                expanded.append(rl)
            else:
                expanded.append(f"{w}-{rl}")
    levels = [
        (l if "-v" in l else build_env_name(ENV_CONFIG["game"], *map(int, l.split("-")), ENV_CONFIG["version"]))
        for l in expanded
    ]
elif levels:
    levels = [
        (l if "-v" in l else build_env_name(ENV_CONFIG["game"], *map(int, l.split("-")), ENV_CONFIG["version"]))
        for l in levels
    ]

# ── EnvConfig ──
env_config = EnvConfig(
    game=ENV_CONFIG["game"],
    world=ENV_CONFIG["world"],
    level=ENV_CONFIG["level"],
    version=ENV_CONFIG["version"],
    movement=ENV_CONFIG["movement"],
    levels=levels,
    worlds=worlds,
    clip_rewards=ENV_CONFIG["clip_rewards"],
    reward_clip_value=ENV_CONFIG["reward_clip_value"],
    stuck_penalty_enabled=ENV_CONFIG["stuck_penalty_enabled"],
    stuck_max_steps=ENV_CONFIG["stuck_max_steps"],
    stuck_penalty=ENV_CONFIG["stuck_penalty"],
    progress_reward_enabled=ENV_CONFIG["progress_reward_enabled"],
    progress_reward_scale=ENV_CONFIG["progress_reward_scale"],
    normalize_observation=ENV_CONFIG["normalize_observation"],
)

# ── AgentConfig ──
agent_config = AgentConfig(**AGENT_CONFIG)

# ── 确定保存根目录 ──
save_root = SAVE_ROOT if FLAT_SAVE_ROOT else compatible_save_root(SAVE_ROOT, env_config)

# ── ArtifactConfig ──
artifact_config = ArtifactConfig(
    save_root=save_root,
    resume=RESUME,
    keep_runs=KEEP_RUNS,
    run_name=RUN_NAME,
    save_every=SAVE_EVERY,
)

# ── TrainingConfig ──
training_config = TrainingConfig(
    episodes=TRAINING_CONFIG["episodes"],
    batch_size=TRAINING_CONFIG["batch_size"],
    burnin=TRAINING_CONFIG["burnin"],
    learn_every=TRAINING_CONFIG["learn_every"],
    sync_every=TRAINING_CONFIG["sync_every"],
    record_every=TRAINING_CONFIG["record_every"],
    eval_every=TRAINING_CONFIG["eval_every"],
    eval_episodes=TRAINING_CONFIG["eval_episodes"],
    num_envs=NUM_ENVS,
    vector=VECTOR_ENV,
)

# ── AppConfig ──
config = AppConfig(
    env=env_config,
    agent=agent_config,
    artifacts=artifact_config,
    training=training_config,
)

# ── 输出信息 ──
active_levels = levels if (levels and len(levels) > 1) else None
print(f"📁 保存目录: {save_root}")
print(f"🎯 关卡: {active_levels or f'{env_config.world}-{env_config.level}'}")
print(f"🧠 智能体: {'NoisyDuelingDQN' if agent_config.noisy_std_init > 0 else 'DuelingDQN + ε-greedy'}")
if active_levels:
    print(f"📋 关卡列表: {active_levels}")

---
## 🖼️ 环境预览

快速验证环境能正常工作。

In [ ]:
# 创建环境并查看观测空间
env = make_mario_env(env_config)
state, info = env.reset()
print(f"观测空间形状: {state.shape}")
print(f"动作空间: {env.action_space}")
print(f"观测值范围: [{state.min():.2f}, {state.max():.2f}]  dtype={state.dtype}")

# 显示一帧
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(state[i], cmap="gray")
    axes[i].set_title(f"Stack [{i}]")
    axes[i].axis("off")
fig.suptitle("环境观测 - 4 帧堆叠 (84×84 灰度)")
plt.tight_layout()
plt.show()
env.close()
del env

---
## 🚀 开始训练

运行下方单元格开始训练。

**训练过程中您可以随时中断 (Kernel → Interrupt)**，下次运行时会自动从最后一个 checkpoint 恢复。

训练日志将保存在 `latest_metrics.csv`，训练曲线图将实时显示。

In [ ]:
# ═══════════════════════════════════════════
#  训练启动
# ═══════════════════════════════════════════

from tqdm import tqdm
from collections import deque

# ── 创建组件 ──
if VECTOR_ENV:
    envs = make_vector_env(NUM_ENVS, env_config)
    states, _ = envs.reset()
    action_dim = envs.single_action_space.n
else:
    env = make_mario_env(env_config)
    action_dim = env.action_space.n

agent = Mario(
    state_dim=env_config.state_dim,
    action_dim=action_dim,
    agent_config=agent_config,
    training_config=training_config,
)

artifacts = create_artifacts(artifact_config)
checkpoint_manager = CheckpointManager(artifact_config, artifacts)
loaded = checkpoint_manager.load_if_available(agent)
logger = MetricLogger(
    artifacts.metrics_csv_paths,
    artifacts.plot_path_groups,
    reset=loaded is None,
)

if loaded:
    print(f"✅ 从 checkpoint 恢复训练 (回合 {agent.loaded_episode}, 步数 {agent.curr_step})")
else:
    print("🆕 从头开始新训练")

start_episode = agent.loaded_episode
total_episodes = training_config.episodes

print(f"🎯 目标回合: {total_episodes}  |  当前进度: {start_episode}/{total_episodes}")

In [ ]:
# ═══════════════════════════════════════════
#  训练主循环
# ═══════════════════════════════════════════
# 如果使用 VECTOR_ENV = True，会自动使用并行训练
# ═══════════════════════════════════════════

if VECTOR_ENV:
    # ────────── 并行训练 ──────────
    completed_episodes = agent.loaded_episode
    next_save_step = ((int(agent.curr_step) // SAVE_EVERY) + 1) * SAVE_EVERY
    ep_rewards = np.zeros(NUM_ENVS, dtype=np.float32)
    ep_lengths = np.zeros(NUM_ENVS, dtype=np.int32)
    ep_max_x_positions = np.zeros(NUM_ENVS, dtype=np.float32)
    ep_action_counts = np.zeros((NUM_ENVS, action_dim), dtype=np.int32)
    recent_losses, recent_qs, recent_rewards = deque(maxlen=100), deque(maxlen=100), deque(maxlen=100)
    best_eval_score = float("-inf")
    progress = tqdm(total=total_episodes, initial=completed_episodes, desc="并行训练", unit="回合")

    try:
        while completed_episodes < total_episodes:
            actions = agent.act_batch(states)
            next_states, rewards, terminated, truncated, infos = envs.step(actions)
            dones = np.logical_or(terminated, truncated)
            replay_next_states = get_replay_next_states(next_states, infos, dones)

            agent.cache_batch(states, replay_next_states, actions, rewards, dones)
            ep_rewards += rewards
            ep_lengths += 1
            ep_max_x_positions = np.maximum(ep_max_x_positions, get_info_array(infos, "max_x_pos", get_info_array(infos, "x_pos", 0.0)))
            for ei, act in enumerate(actions):
                ep_action_counts[ei, act] += 1

            q, loss = agent.learn()
            if loss is not None:
                recent_losses.append(loss)
                recent_qs.append(q)

            if should_save_step(agent, next_save_step):
                checkpoint_manager.save(agent)
                next_save_step = ((int(agent.curr_step) // SAVE_EVERY) + 1) * SAVE_EVERY

            for ei, done in enumerate(dones):
                if not done:
                    continue
                completed_episodes += 1
                agent.current_episode = completed_episodes
                episode_reward = float(ep_rewards[ei])
                recent_rewards.append(episode_reward)
                ep_len = int(ep_lengths[ei])
                noop_rate = float(ep_action_counts[ei, 0]) / max(1, ep_len)
                mua = int(np.argmax(ep_action_counts[ei]))
                logger.log_episode_metrics(
                    episode_reward, ep_len,
                    rolling_mean(list(recent_losses)), rolling_mean(list(recent_qs)),
                    float(ep_max_x_positions[ei]), noop_rate, mua,
                )
                progress.update(1)

                ep_rewards[ei] = 0.0
                ep_lengths[ei] = 0
                ep_max_x_positions[ei] = 0.0
                ep_action_counts[ei] = 0

                # record() 始终写入 CSV，emit=True 时才打印+绘图
                emit_record = should_record(completed_episodes, training_config.record_every)
                logger.record(
                    episode=completed_episodes,
                    epsilon=agent.exploration_rate,
                    step=agent.curr_step,
                    emit=emit_record,
                )
                if emit_record:
                    checkpoint_manager.save(agent, episode=completed_episodes)

                if should_eval(completed_episodes, training_config.eval_every):
                    result = evaluate_agent(agent, config)
                    tqdm.write(f"📊 评估 | 分数={result.score:.1f} 平均奖励={result.mean_reward:.2f} 最大x={result.mean_max_x_pos:.0f} 通关率={result.flag_rate:.2f}")
                    if result.score > best_eval_score:
                        best_eval_score = result.score
                        checkpoint_manager.save_best(agent, eval_score=result.score, eval_reward=result.mean_reward, eval_max_x_pos=result.mean_max_x_pos, eval_flag_rate=result.flag_rate, episode=completed_episodes)

                if completed_episodes >= total_episodes:
                    break

            progress.set_postfix({
                "步数": agent.curr_step,
                "探索率": f"{agent.exploration_rate:.3f}",
                "近100轮奖励": f"{rolling_mean(list(recent_rewards)):.1f}" if recent_rewards else "N/A",
            })
            states = next_states

        checkpoint_manager.save(agent, episode=completed_episodes)
    except KeyboardInterrupt:
        tqdm.write("⏹️ 训练被中断，资源已清理。下次运行会自动恢复。")
    finally:
        progress.close()
        close_vector_envs(envs, terminate=True)

else:
    # ────────── 单环境训练 ──────────
    completed_episodes = agent.loaded_episode
    next_save_step = ((int(agent.curr_step) // SAVE_EVERY) + 1) * SAVE_EVERY
    best_eval_score = float("-inf")
    progress = tqdm(total=total_episodes, initial=completed_episodes, desc="单环境训练", unit="回合")

    try:
        while completed_episodes < total_episodes:
            # 多关卡：每次随机换关卡
            if active_levels:
                level_name = random.choice(active_levels)
                env.close()
                env = make_mario_env(env_config, level_name=level_name)

            state, _ = env.reset()
            ep_reward = 0.0
            ep_length = 0
            ep_max_x_pos = 0.0
            action_counts = np.zeros(action_dim, dtype=np.int32)
            last_loss = None
            last_q = None

            while True:
                action = agent.act(state)
                next_state, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                action_counts[action] += 1
                agent.cache(state, next_state, action, reward, done)
                q, loss = agent.learn()
                logger.log_step(reward, loss, q)
                ep_reward += reward
                ep_length += 1
                ep_max_x_pos = max(ep_max_x_pos, info.get("max_x_pos", info.get("x_pos", 0.0)))
                if loss is not None:
                    last_loss = loss
                    last_q = q

                if should_save_step(agent, next_save_step):
                    checkpoint_manager.save(agent)
                    next_save_step = ((int(agent.curr_step) // SAVE_EVERY) + 1) * SAVE_EVERY

                state = next_state
                if done or info.get("flag_get"):
                    break

            completed_episodes += 1
            agent.current_episode = completed_episodes
            noop_rate = float(action_counts[0]) / max(1, ep_length)
            most_used_action = int(np.argmax(action_counts))
            logger.log_episode(max_x_pos=ep_max_x_pos, noop_rate=noop_rate, most_used_action=most_used_action)
            progress.update(1)
            progress.set_postfix({
                "步数": agent.curr_step,
                "探索率": f"{agent.exploration_rate:.3f}",
                "回合奖励": f"{ep_reward:.1f}",
                "回合长度": ep_length,
                "损失": "-" if last_loss is None else f"{last_loss:.4f}",
                "Q值": "-" if last_q is None else f"{last_q:.4f}",
            })

            # record() 始终写入 CSV，emit=True 时才打印+绘图
            emit_record = should_record(completed_episodes, training_config.record_every)
            logger.record(
                episode=completed_episodes,
                epsilon=agent.exploration_rate,
                step=agent.curr_step,
                emit=emit_record,
            )
            if emit_record:
                checkpoint_manager.save(agent, episode=completed_episodes)

            if should_eval(completed_episodes, training_config.eval_every):
                result = evaluate_agent(agent, config)
                tqdm.write(f"📊 评估 | 分数={result.score:.1f} 平均奖励={result.mean_reward:.2f} 最大x={result.mean_max_x_pos:.0f} 通关率={result.flag_rate:.2f}")
                if result.score > best_eval_score:
                    best_eval_score = result.score
                    checkpoint_manager.save_best(agent, eval_score=result.score, eval_reward=result.mean_reward, eval_max_x_pos=result.mean_max_x_pos, eval_flag_rate=result.flag_rate, episode=completed_episodes)

        checkpoint_manager.save(agent, episode=completed_episodes)
    except KeyboardInterrupt:
        tqdm.write("⏹️ 训练被中断，下次运行会自动恢复。")
    finally:
        progress.close()
        if 'env' in dir():
            env.close()

print(f"\n{'='*50}")
print(f"🏁 训练完成！共 {completed_episodes} 回合, {agent.curr_step} 步")
print(f"📁 Checkpoint 保存在: {save_root}")

---
## 📈 训练指标可视化

运行此单元格查看训练过程中记录的指标曲线。曲线会在每轮 `record_every` 回合自动更新，也可以随时运行此单元格查看最新状态。

In [ ]:
# 检查是否有指标数据
if logger.moving_avgs[0]:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    titles = ["平均奖励", "平均回合长度", "平均损失", "平均 Q 值"]
    ylabels = ["奖励", "长度", "损失", "Q 值"]
    colors = ["#2ecc71", "#3498db", "#e74c3c", "#9b59b6"]

    for i, (ax, title, ylabel, color) in enumerate(zip(axes.flat, titles, ylabels, colors)):
        episodes = range(1, len(logger.moving_avgs[i]) + 1)
        ax.plot(episodes, logger.moving_avgs[i], color=color, linewidth=2)
        ax.set_title(title, fontsize=14, fontweight="bold")
        ax.set_xlabel("回合", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.grid(True, linestyle="--", alpha=0.5)

        # 双轴：显示最大 x 位置和 NOOP 率
        if i == 0:
            ax2 = ax.twinx()
            if logger.max_x_pos_avgs:
                ax2.plot(range(1, len(logger.max_x_pos_avgs) + 1), logger.max_x_pos_avgs, color="#e67e22", linewidth=1.5, linestyle="--", alpha=0.7, label="最大 x 位置")
            ax2.set_ylabel("最大 x 位置", fontsize=11, color="#e67e22")
            ax2.tick_params(axis="y", labelcolor="#e67e22")
            ax2.legend(loc="upper right", fontsize=10)

    plt.suptitle("训练曲线", fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.show()

    # 显示最新指标
    print(f"\n{'='*60}")
    print(f"📊 最新指标 (最近 {len(logger.ep_rewards)} 回合)")
    print(f"{'='*60}")
    print(f"  平均奖励:        {logger.moving_avgs[0][-1]:.3f}")
    print(f"  平均长度:        {logger.moving_avgs[1][-1]:.3f}")
    print(f"  平均损失:        {logger.moving_avgs[2][-1]:.3f}  " if logger.moving_avgs[2][-1] > 0 else f"  平均损失:        {logger.moving_avgs[2][-1]:.3f}")
    print(f"  平均 Q 值:       {logger.moving_avgs[3][-1]:.3f}")
    print(f"  平均最大 x:      {logger.max_x_pos_avgs[-1]:.1f}" if logger.max_x_pos_avgs else "")
    print(f"  平均 NOOP 率:    {logger.noop_rate_avgs[-1]:.3f}" if logger.noop_rate_avgs else "")
    print(f"  探索率:          {agent.exploration_rate:.4f}")
    print(f"  总步数:          {agent.curr_step}")
else:
    print("⚠️ 还没有指标数据，请先运行训练单元格。")

---
## 🧪 评估智能体

对当前智能体进行贪心评估（关闭探索），测试其真实水平。

In [ ]:
print("🧪 开始评估...")
result = evaluate_agent(agent, config)

print(f"\n{'='*50}")
print(f"📊 评估结果")
print(f"{'='*50}")
print(f"  🏆 综合分数:     {result.score:.1f}")
print(f"  💰 平均奖励:     {result.mean_reward:.2f}")
print(f"  📏 平均长度:     {result.mean_length:.1f}")
print(f"  ➡️  平均最大 x:  {result.mean_max_x_pos:.0f}")
print(f"  🚩 通关率:       {result.flag_rate:.2%}")
print(f"{'='*50}")

---
## 🎬 观看智能体玩游戏

运行此单元格观看 Mario 在当前关卡中的表现。（如果 checkpoints 中有训练好的模型，会自动加载最佳模型）

In [ ]:
# 关闭探索，纯贪心模式
agent.exploration_rate = 0.0
agent.exploration_rate_min = 0.0

play_env = make_mario_env(env_config, render_mode="rgb_array")
state, _ = play_env.reset()
total_reward = 0.0
step = 0
flag_get = False

plt.ion()
fig, ax = plt.subplots(figsize=(8, 8))
img_display = None
ax.axis("off")

try:
    for _ in range(10000):  # 安全上限
        frame = play_env.render()
        if img_display is None:
            img_display = ax.imshow(frame)
        else:
            img_display.set_data(frame)
        display.display(fig)
        display.clear_output(wait=True)

        action = agent.act_eval(state)
        state, reward, terminated, truncated, info = play_env.step(action)
        total_reward += reward
        step += 1

        if terminated or truncated or info.get("flag_get"):
            flag_get = bool(info.get("flag_get", False))
            print(f"\n🎬 游戏结束 | 步数={step}  奖励={total_reward:.1f}  通关={'✅' if flag_get else '❌'}")
            break
finally:
    play_env.close()
    plt.ioff()
    plt.close()

---
## 🔄 断点续训

如果训练中断了，只需确保上方的 `RESUME = True`，然后重新运行**训练单元格**即可自动从最近 checkpoint 恢复。

如果您想手动指定一个 checkpoint 文件来恢复：
1. 设置下方 `MANUAL_CHECKPOINT` 为 checkpoint 文件路径
2. 运行此单元格加载模型
3. 再运行训练单元格继续训练

In [ ]:
MANUAL_CHECKPOINT = None  # 例如: Path("checkpoints/1-1-v0_right_only/best.chkpt")

if MANUAL_CHECKPOINT and MANUAL_CHECKPOINT.exists():
    checkpoint_manager = CheckpointManager(artifact_config, artifacts)
    agent = Mario(
        state_dim=env_config.state_dim,
        action_dim=action_dim,
        agent_config=agent_config,
        training_config=training_config,
    )
    loaded = checkpoint_manager.load(agent, MANUAL_CHECKPOINT)
    if loaded:
        print(f"✅ 已加载: {MANUAL_CHECKPOINT}")
        print(f"   回合: {agent.loaded_episode}, 步数: {agent.curr_step}")
    else:
        print("❌ 加载失败")
else:
    print("ℹ️ 未指定手动 checkpoint，使用自动续训（需 RESUME=True）")

---
## 💾 导出模型

将训练好的模型导出为 PyTorch state dict（`.pt` 文件），方便后续推理部署。

In [ ]:
export_path = Path("checkpoints") / f"mario_model_{agent.curr_step}.pt"
export_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(agent.net.state_dict(), export_path)
print(f"✅ 模型已导出到: {export_path.resolve()}")
print(f"   步数: {agent.curr_step}")
print(f"   网络结构: {type(agent.net.online).__name__}")
print(f"   动作数: {agent.action_dim}")

---
## 📋 附录：快速参数参考

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `movement` | `right_only` | 动作空间：`right_only`(5个) / `simple`(7个) / `complex`(12个) |
| `noisy_std_init` | `0.5` | NoisyNet 噪声强度。设为 `0` 则使用 epsilon-greedy 探索 |
| `n_step` | `3` | N-step 自举步数，越大则 credit assignment 越远 |
| `gamma` | `0.99` | 折扣因子，越高越重视远期奖励 |
| `per_alpha` | `0.6` | 优先级采样强度，`0` = 均匀采样 |
| `clip_rewards` | `True` | 裁剪奖励到 [-1, 1] 有助于训练稳定性 |
| `stuck_penalty_enabled` | `True` | 卡住 120 步自动终止并扣分，防止无限循环 |
| `progress_reward_enabled` | `True` | 向右走的额外奖励，引导向右探索 |

**典型配置组合：**

| 场景 | 配置 |
|------|------|
| 快速验证 | 单环境, right_only, 200 回合, burnin=10000, eval_every=50 |
| 正式训练 | 并行(8 env), simple, NoisyNet, 20000 回合 |
| 多关卡 | levels=["1-1","1-2","1-3"], 并行, no_reward_clip (奖励值跨关卡可能有差异) |
| 无 NoisyNet | noisy_std_init=0 (回退到 epsilon-greedy) |